# Reproduce the solar filament predictions

This notebook runs the selected model on the official GONG H-alpha test observations and writes the exact competition CSV. It uses the included frozen checkpoint; training is unnecessary.

Use Python 3.12 and install the repository's pinned requirements in this kernel. Run from the repository root and place the official competition files under the `data/` directory. Do not use the public MAGFiLO distribution as a substitute.

1. Verify the release checkpoint, config and test file identities.
2. Run the same native image reconstruction and export used for the selected artifact.
3. Decode and audit the written CSV against all input images and compare its hash.


In [ ]:
import hashlib
import json
import os
from pathlib import Path
import subprocess
import sys
from tempfile import mkdtemp

root = Path.cwd()
release = json.loads((root / "artifacts" / "release.json").read_text(encoding="utf-8"))
test_images = root / "data" / "MAGFiLO_1.0_Kaggle_2026" / "test" / "test_images"
out_root = root / "output"
out_root.mkdir(parents=True, exist_ok=True)
out = Path(mkdtemp(prefix="notebook-reproduction-", dir=out_root))
env = os.environ.copy()
env.update({"PYTHONPATH": str(root / "src"), "OMP_NUM_THREADS": "1", "MKL_NUM_THREADS": "1"})
def sha256(path):
    with path.open("rb") as f:
        return hashlib.file_digest(f, "sha256").hexdigest()
def run(args):
    subprocess.run([sys.executable, *map(str, args)], cwd=root, env=env, check=True)


## Verify the inputs

The release manifest lists the model, settings and hashes. The test file manifest is an inventory of the organizer-supplied input JPEGs; it does not contain labels or predicted masks.


In [ ]:
for name, digest in release["artifact_sha256"].items():
    assert sha256(root / name) == digest, f"Unexpected file: {name}"
expected = json.loads((root / release["test_manifest"]).read_text(encoding="utf-8"))
actual = {p.name: sha256(p) for p in sorted(test_images.glob("*.jpeg"))}
assert actual == expected, "Official test images are missing, extra or different."
print(f"Verified {len(actual)} test images and the frozen release files.")


## Run the frozen pipeline

The pipeline normalizes each grayscale disk, predicts overlapping tiles, reconstructs a native 2048 x 2048 probability map, and converts the selected postprocessing result to instances. It writes compressed COCO RLE for each predicted mask and an image-level coverage receipt. Use the release device to compare exact bytes; different hardware and library builds can change results around the probability threshold.


In [ ]:
csv_path = out / "submission.csv"
coverage_path = out / "coverage.json"
receipt_path = out / "inference.json"
run(["-m", "filament", "infer", "--config", root / release["config"],
     "--checkpoint", root / release["checkpoint"], "--images", test_images,
     "--output", csv_path, "--coverage", coverage_path, "--receipt", receipt_path,
     "--device", release["device"], *release.get("extra_inference_args", [])])


## Independently decode the export

The reviewer checks ID coverage, native dimensions, mask orientation and per-instance placement. A disk with zero predictions has no CSV rows; the coverage record still identifies the disk, and the release manifest verifies its input hash.


In [ ]:
qa_path = out / "qa.json"
run(["qa/verify_entry.py", "export", "--csv", csv_path, "--coverage", coverage_path,
     "--images", test_images, "--output", qa_path])
observed = sha256(csv_path)
print(f"CSV SHA-256: {observed}")
assert observed == release["submission_sha256"], "Export is structurally valid but differs from the frozen artifact."
print(f"Reproduced {csv_path} ({csv_path.stat().st_size:,} bytes).")


## Validation scope

The checks here establish input identity, native mask encoding and reproducibility. Competition scoring requires the organizer's hidden annotations and is separate from this notebook.
